# K-IFRS Parent-Child 통합 검색 테스트

Child 검색 → Parent heading 조회 → 형제 Child 묶기 검증

- **Retriever**: QdrantDenseRetriever (Upstage `solar-embedding-1-large`)
- **검색 방식**: Child 벡터 검색 → Parent heading 매핑 → 같은 Parent 아래 Sibling 조회
- **목적**: 문단 검색 결과가 Parent 단위로 올바르게 묶이는지, 형제 문단이 정상 조회되는지 확인

In [1]:
import os

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from qdrant_client import QdrantClient

from search.config import QDRANT_PATH, CHILD_COLLECTION, PARENT_COLLECTION, MODEL_NAME
from search.retriever import search_with_parent

load_dotenv()
print("패키지 로드 완료")

c:\Study\_database\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


패키지 로드 완료


## 1. 설정

In [2]:
TOP_K = 5  # Child 검색 상위 결과 수

QUERIES = [
    "유형자산 감가상각 방법",
    "원가모형과 재평가모형의 차이",
    "유형자산 제거 시 손익 처리",
]

print(f"TOP_K: {TOP_K}")
print(f"테스트 쿼리: {len(QUERIES)}개")

TOP_K: 5
테스트 쿼리: 3개


## 2. Qdrant & 임베딩 모델 초기화

In [3]:
print(f"[1/2] Upstage {MODEL_NAME} 초기화 중...")
embeddings = UpstageEmbeddings(
    model=MODEL_NAME,
    upstage_api_key=os.getenv("UPSTAGE_API_KEY"),
)

print("[2/2] Qdrant 연결 중...")
client = QdrantClient(path=QDRANT_PATH)

child_count = client.count(CHILD_COLLECTION).count
parent_count = client.count(PARENT_COLLECTION).count
print(f"  Child 포인트: {child_count}개")
print(f"  Parent 포인트: {parent_count}개")
print("초기화 완료")

[1/2] Upstage solar-embedding-1-large 초기화 중...
[2/2] Qdrant 연결 중...
  Child 포인트: 12313개
  Parent 포인트: 2231개
초기화 완료


## 3. Parent-Child 통합 검색 테스트

`search_with_parent` 함수의 동작:
1. 쿼리를 임베딩하여 **Child 컬렉션**에서 벡터 검색 (Top-K)
2. 각 Child의 `parent_id`로 **Parent 컬렉션**에서 heading 조회
3. 같은 `parent_id`를 가진 **형제 Child(Siblings)**를 모두 조회
4. Parent 단위로 그룹핑하여 반환

이를 통해 검색된 문단의 **맥락(같은 섹션의 전후 문단)**을 함께 확인할 수 있습니다.

In [4]:
from IPython.display import Markdown, display


def display_search_results(query: str, groups: list[dict]):
    """검색 결과를 Markdown으로 포맷팅하여 출력"""
    lines = [f"### 쿼리: {query}\n"]

    if not groups:
        lines.append("검색 결과 없음\n")
        display(Markdown("\n".join(lines)))
        return

    for g_idx, g in enumerate(groups, 1):
        lines.append(f"#### [{g_idx}] Parent: {g['heading']}")
        lines.append(f"> `parent_id`: {g['parent_id']}\n")

        # 매칭된 Child
        lines.append("**Matched Children:**\n")
        for mc in g["matched_children"]:
            preview = mc["content"][:200].replace("\n", " ")
            lines.append(
                f"- 문단 **{mc['para_number']}** (score: {mc['score']:.4f})  \n"
                f"  `{preview}...`\n"
            )

        # 형제 Child
        matched_ids = {mc["chunk_id"] for mc in g["matched_children"]}
        other_siblings = [s for s in g["siblings"] if s["chunk_id"] not in matched_ids]

        if other_siblings:
            lines.append(f"**Siblings ({len(other_siblings)}건):**\n")
            for s in other_siblings:
                preview = s["content"][:100].replace("\n", " ")
                lines.append(f"- 문단 {s['para_number']}: `{preview}...`")
        else:
            lines.append("**Siblings:** 형제 없음 (단독 문단)")

        lines.append("\n---\n")

    display(Markdown("\n".join(lines)))


print("display_search_results 정의 완료")

display_search_results 정의 완료


In [5]:
all_results = {}

for q in QUERIES:
    print(f"검색 중: {q}")
    groups = search_with_parent(client, embeddings, q, top_k=TOP_K)
    all_results[q] = groups
    display_search_results(q, groups)

print(f"\n전체 {len(QUERIES)}개 쿼리 검색 완료")

검색 중: 유형자산 감가상각 방법


### 쿼리: 유형자산 감가상각 방법

#### [1] Parent: 인식시점 이후의 측정
> `parent_id`: KIFRS1016_main_h_인식시점_이후의_측정

**Matched Children:**

- 문단 **62** (score: 0.5487)  
  `62 유형자산의 감가상각대상금액을 내용연수 동안 체계적으로 배부  하기 위해 다양한 방법을 사용할 수 있다 . 이러한 감가상각방법에  는 정액법, 체감잔액법과 생산량비례법이 있다 . 정액법은 잔존가  치가 변동하지 않는다고 가정할 때 자산의 내용연수 동안 매 기  간 일정액의 감가상각액을 계상하는 방법이다 . 체감잔액법은 자산  의 내용연수 동안 감가상각액...`

- 문단 **1002** (score: 0.4935)  
  `1002 호 참조 ), 개발활동에 사용되는 유형자산의 감가상각액은 기  업회계기준서 제 1038 호 ‘ 무형자산 ’ 에 따라 해당 무형자산의 원가  에 포함될 수 있다 .  감가상각대상금액과 감가상각기간  유형자산의 감가상각대상금액은 내용연수에 걸쳐 체계적인 방법  으로 배분한다  유형자산의 잔존가치와 내용연수는 적어도 매 회계연도말에 재검  토한다 재검토...`

- 문단 **45** (score: 0.4817)  
  `45 유형자산을 구성하고 있는 유의적인 부분에 해당 유형자산의 다  른 유의적인 부분과 동일한 내용연수 및 감가상각방법을 적용하  는 수가 있다 . 이러한 경우에는 감가상각액을 결정할 때 하나의  집단으로 통합할 수 있다 ....`

- 문단 **59** (score: 0.4684)  
  `59 토지의 원가에 해체, 제거 및 복구원가가 포함된 경우에는 그러한  원가를 관련 경제적효익이 유입되는 기간에 감가상각한다 . 경우에  따라서는 토지의 내용연수가 한정될 수 있다 . 이 경우에는 관련                   - 25  경제적효익이 유입되는 형태를 반영하는 방법으로 토지를 감가상  각한다 .  감가상각방법  유형자산의 감가상각방법은...`

**Siblings (20건):**

- 문단 None: `를 회계정책으로 선택하여 유형자산의 유형별로 동일하게 적용한  다  29A 일부 기업은 투자자에게 펀드 내의 단위별로 결정되는 이익을 제  공하는 투자펀드를 내부적으로 또는 외부적...`
- 문단 34: `34 재평가의 빈도는 재평가되는 유형자산의 공정가치 변동에 따라  달라진다 . 재평가된 자산의 공정가치가 장부금액과 중요하게 차이  가 나는 경우에는 추가적인 재평가가 필요하다 ....`
- 문단 35: `35 유형자산을 재평가할 때, 그 자산의 장부금액을 재평가금액으로  조정한다 . 재평가일에 그 자산을 다음 중 하나의 방법으로 회계처  리한다 .  ⑴ 자산 장부금액의 재평가와 일...`
- 문단 37: `37 유형자산의 유형은 기업의 영업에서 특성과 용도가 비슷한 자산  의 집합이다 . 다음은 개별 유형의 예이다 .  ⑴ 토지  ⑵ 토지와 건물  ⑶ 기계장치  ⑷ 선박       ...`
- 문단 38: `38 유형자산별로 선택적 재평가를 하거나 서로 다른 기준일의 평가  금액이 혼재된 재무보고를 하는 것을 방지하기 위하여 동일한 유  형 내의 유형자산은 동시에 재평가한다 . 그러나...`
- 문단 41: `41 어떤 유형자산 항목과 관련하여 자본에 계상된 재평가잉여금은  그 자산이 제거될 때 이익잉여금으로 직접 대체할 수 있다 . 자산  이 폐기되거나 처분될 때에 재평가잉여금 전부를...`
- 문단 42: `42 유형자산의 재평가로 인한 법인세효과는 기업회계기준서 제 1012  호 ‘ 법인세 ' 에 따라 인식하고 공시한다 .  감가상각  유형자산을 구성하는 일부의 원가가 당해 유형자산...`
- 문단 44: `44 유형자산의 원가는 그 유형자산을 구성하고 있는 유의적인 부분  에 배분하여 각 부분별로 감가상각한다 . 예를 들면, 항공기 동체  와 엔진을 별도로 구분하여 감가상각하는 것이...`
- 문단 46: `46 유형자산의 일부를 별도로 구분하여 감가상각하는 경우에는 동일  한 유형자산을 구성하고 있는 나머지 부분도 별도로 구분하여 감  가상각한다 . 나머지 부분은 개별적으로 유의적이...`
- 문단 47: `47 유형자산의 전체원가에 비교하여 해당 원가가 유의적이지 않은  부분도 별도로 분리하여 감가상각할 수 있다 .  각 기간의 감가상각액은 다른 자산의 장부금액에 포함되는 경우  가...`
- 문단 49: `49 각 기간의 감가상각액은 일반적으로 당기손익으로 인식한다 . 그러  나 유형자산에 내재된 미래경제적효익이 다른 자산을 생산하는  데 사용되는 경우도 있다 . 이 경우 유형자산의...`
- 문단 52: `52 유형자산의 공정가치가 장부금액을 초과하더라도 잔존가치가 장  부금액을 초과하지 않는 한 감가상각액을 계속 인식한다 . 유형자                   - 23  산을 ...`
- 문단 53: `53 감가상각대상금액은 유형자산의 원가에서 잔존가치를 차감한 금  액이다 . 실무적으로 잔존가치는 경미한 경우가 많으므로 감가상각  대상금액을 계산할 때 중요하게 다루어지지 않는다...`
- 문단 54: `54 유형자산의 잔존가치는 해당 자산의 장부금액과 같거나 큰 금액  으로 증가할 수도 있다 . 이 경우에는 자산의 잔존가치가 장부금액  보다 작은 금액으로 감소될 때까지는 유형자산...`
- 문단 55: `55 유형자산의 감가상각은 자산이 사용가능한 때부터 시작한다 . 즉,  경영진이 의도하는 방식으로 자산을 가동하는 데 필요한 장소와  상태에 이른 때부터 시작한다 . 감가상각은 기...`
- 문단 56: `56 유형자산의 미래경제적효익은 주로 사용함으로써 소비하는 것이  일반적이다 . 그러나 자산을 사용하지 않더라도 기술적 또는 상업  적 진부화와 마모 또는 손상 등의 다른 요인으로...`
- 문단 57: `57 유형자산의 내용연수는 자산으로부터 기대되는 효용에 따라 결정  된다 . 유형자산은 기업의 자산관리정책에 따라 특정기간이 경과되  거나 자산에 내재하는 미래경제적효익의 특정부분...`
- 문단 58: `58 토지와 건물을 동시에 취득하는 경우에도 이들은 분리가능한 자  산이므로 별개의 자산으로 회계처리한다 . 채석장이나 매립지 등을  제외하고는 토지는 내용연수가 무한하므로 감가상...`
- 문단 66: `66 유형자산과 관련된 손상차손이나 기타 손실, 제 3 자에 대한 보상청  구나 그 보상금의 수령 그리고 대체 유형자산의 매입이나 건설은  각각 구분되는 경제적 사건이므로 다음과 ...`
- 문단 1036: `1036 호를 적용한다 . 기업회계기준서 제 1036 호는 자산의 장부금액  의 검토방법, 자산의 회수가능액의 결정방법 및 손상차손과 손상  차손환입의 인식시기를 설명하고 있다 ....`

---

#### [2] Parent: 이 기준서의 주요 특징
> `parent_id`: KIFRS1016_bc_h_이_기준서의_주요_특징

**Matched Children:**

- 문단 **None** (score: 0.4974)  
  `이 기준서는 유형자산에 대한 회계처리와 공시에 대한 사항을 정하고 있다 .  적용범위  이 기준서는 적용범위에서 제외되는 일부 유형자산을 제외한 대부분의  유형자산에 대한 회계처리에 적용한다 . 다만, 다른 한국채택국제회계기준  서에서 상이한 회계처리를 요구하거나 허용하는 경우에는 이 기준서를  적용하지 아니한다 .  인식  유형자산으로 인식되기 위해서는 ...`

**Siblings:** 형제 없음 (단독 문단)

---


검색 중: 원가모형과 재평가모형의 차이


### 쿼리: 원가모형과 재평가모형의 차이

#### [1] Parent: 후속측정
> `parent_id`: KIFRS1040_bc_h_후속측정

**Matched Children:**

- 문단 **None** (score: 0.4514)  
  `회계모형  B43 IAS 25 에 따라 투자부동산에 대해 다양한 회계처리 중에서 기업  이 선택할 수 있도록 허용되었다 . (IAS 16 의 원칙적 회계처리방법  인 상각후원가, IAS 16 에서 허용된 대체적 회계처리방법인 상각후  재평가, IAS 25 의 손상을 차감한 원가 또는 IAS 25 에 따른 재평  가 ). [10)]  9) 2003 년에 IA...`

**Siblings:** 형제 없음 (단독 문단)

---

#### [2] Parent: 과의 비교
> `parent_id`: KIFRS2101_bc_h_과의_비교

**Matched Children:**

- 문단 **BC25** (score: 0.4482)  
  `BC25 해석 공개초안에 대한 여러 의견제출자들은 이 해석서를 재평가  된 자산들에 어떻게 적용하여야 하는지를 명확하게 해줄 것을 요  청하였다 . IFRIC 은 다음과 같은 점에 주목하였다 .  ⑴ 기업이 재평가모형을 선택한다면 IAS 16 에 따라 장부금액이  대차대조표일 [3)] 의 공정가치를 이용하여 결정되었을 금액과 중  요하게 다르지 않도록 그 ...`

- 문단 **BC24** (score: 0.4391)  
  `BC24 IAS 16 은 기업이 유형별로 원가모형이나 재평가모형을 선택하여  유형자산을 측정하는 것을 허용한다 . IFRIC 의 관점에 따르면 기업  이 IAS 16 에 따라 선택한 측정모형은 이 해석서에 의해 영향 받  지 않을 것이다 .                    - 31...`

**Siblings (5건):**

- 문단 BC19: `BC19 결론을 도출하면서, IFRIC 은 SFAS 143 ' 퇴역된 자산과 관련된 의  무에 대한 회계처리 (Accounting for Asset Retirement  Oblig...`
- 문단 BC20: `BC20 이 해석서에서 요구되는 추정현금흐름의 변경에 관한 회계처리는  US GAAP 과 일관되는데, D2 의 제안의 경우에는 일관되지 않았다 .  그러나 IFRIC 은 IAS 3...`
- 문단 BC21: `BC21 이 해석서를 마련하면서, IFRIC 은 IASB 의 IAS 16 에 대한 개정을  고려하였으며, 이 개정이 이 해석서와 IAS 16 의 상호작용을 설명  해 줄 것이라고 ...`
- 문단 BC22: `BC22 IAS 16(2003 년 전면개정 ) 은 유형자산을 해체, 제거하거나 그 자산  이 위치한 부지를 복구하여야 하는 의무가 그 유형자산을 취득한  시점에 발생하거나 그 유형...`
- 문단 BC23: `BC23 그러나 IAS 16 에 대한 개선을 고려하면서 IASB 는 기업이 ⑴ 인식  된 의무의 최초 추정금액의 변경, ⑵ 인식된 의무의 증가 영향이  나 인식된 의무에 대한 이자...`

---

#### [3] Parent: 인식 후의 측정
> `parent_id`: KIFRS1038_main_h_인식_후의_측정

**Matched Children:**

- 문단 **73** (score: 0.4446)  
  `73 무형자산의 유형은 기업의 영업에서 특성과 용도가 비슷한 자산의  집합이다 . 자산을 선택적으로 재평가하거나 재무제표에서 서로 다  른 기준일의 원가와 가치가 혼재된 금액을 보고하는 것을 방지하기  2) 이 기준서의 ‘재평가’는 ‘감정평가 및 감정평가사에 관한 법률’ 등에서 언급하는 ‘재평가’와 그 용도, 대상 및  구체적 평가방법 등에서 서로 다른 의...`

**Siblings (9건):**

- 문단 None: `계처리하는 경우에는 같은 유형의 기타 모든 자산도 그에 대한 활  성시장이 없는 경우를 제외하고는 동일한 방법을 적용하여 회계처  리한다...`
- 문단 76: `76 재평가모형을 적용하는 경우에 다음 사항을 허용하지 않는다 .  ⑴ 이전에 자산으로 인식하지 않은 무형자산의 재평가  ⑵ 원가가 아닌 금액으로 무형자산을 최초로 인식...`
- 문단 77: `77 재평가모형은 자산을 원가로 최초에 인식한 후에 적용한다 . 그러나  일부 과정이 종료될 때까지 인식기준을 충족하지 않아서 무형자산  의 원가의 일부만 자산으로 인식한 경우 (...`
- 문단 78: `78 무형자산에 대하여 활성시장이 존재하는 것이 흔하지는 않다 . 예를  들면, 어떤 국가에서는 자유롭게 양도가 가능한 택시 라이선스, 어  업권이나 생산할당량에 대하여 활성시장이...`
- 문단 79: `79 재평가의 빈도는 재평가되는 무형자산의 공정가치의 변동성에 따라  달라진다 . 재평가된 자산의 공정가치가 장부금액과 중요하게 차이  가 나는 경우에는 추가적인 재평가가 필요하다...`
- 문단 80: `80 무형자산을 재평가할 때그 자산의 장부금액을 재평가금액으로 조  정한다 . 재평가일에 그 자산을 다음 중 하나의 방법으로 회계처리  한다 .  ⑴ 자산 장부금액의 재평가와 일치...`
- 문단 83: `83 재평가한 무형자산에 대하여 더 이상 활성시장이 존재하지 않는다  는 것은 자산이 손상되어 기업회계기준서 제 1036 호 ‘ 자산손상 ’ 에 따  라 손상검사를 할 필요가 있다...`
- 문단 84: `84 자산의 공정가치를 이후의 측정일에 활성시장을 기초로 하여 측정  할 수 있는 경우에는 그 날부터 재평가모형을 적용한다 .  무형자산의 장부금액이 재평가로 인하여 증가된 경우에...`
- 문단 87: `87 자본에 포함된 재평가잉여금 누계액은 그 잉여금이 실현되는 시점  에 이익잉여금으로 직접 대체할 수 있다 . 자산의 폐기나 처분 시점  에 전체 잉여금이 실현될 수 있다 . 그...`

---

#### [4] Parent: 예상되는 개정 영향에 대한 분석
> `parent_id`: KIFRS1016_bc_h_예상되는_개정_영향에_대한_분석

**Matched Children:**

- 문단 **BC107** (score: 0.4422)  
  `BC107 현행 IFRS 채택기업이 생산용식물에 IAS 16 의 원가모형을 적용하  기로 선택했다고 가정할 때 주요 변화는 다음과 같을 것이다  |영향|IAS 41의<br>공정가치모형|IAS 16의<br>원가모형|영향| |---|---|---|---| |재무상태|(생산물과 함<br>께)<br>순공정가<br>치로 측정|원가에서 감<br>가상각누계액<br>과 ...`

**Siblings (18건):**

- 문단 BC99: `BC99 다음 문단에서는 생산용식물 회계처리 규정의 개정에 따라 예상  되는 영향에 대한 IASB 의 분석을 기술한다 ....`
- 문단 BC100: `BC100 IASB 는 새로운 규정을 시행하는 데 예상되는 원가와 제정되거나  개정되는 각 기준서를 계속 적용하는 데 예상되는 원가 및 효익  ( 원가와 효익은 집합적으로 ‘ 영향...`
- 문단 BC101: `BC101 IASB 는 제안 내용의 공식적인 의견조회, 현장 연구, 분석, 의견수  집활동을 통한 이해관계자와의 협의를 거쳐 제정되거나 개정되는  기준서 제안 내용의 예상 영향에 ...`
- 문단 BC102: `BC102 IASB 는 개정 내용의 예상 영향을 평가할 때 다음의 논점을 고려  하였다 ( 문단 BC106~BC117 참조 ).  ⑴ IFRS 를 적용하는 작성자의 재무제표에서 변...`
- 문단 BC103: `BC103 이 개정에서는 기업이 생산용식물에 IAS 16 에 따라 원가모형이나  재평가모형 가운데 어느 하나를 적용하도록 허용한다 . IASB 는 대  부분의 기업이 다음과 같은 ...`
- 문단 BC104: `BC104 따라서 문단 BC106~BC117 의 예상 영향 분석에서는 IAS 41 의 공  정가치모형과 비교하여 IAS 16 의 원가모형을 적용할 때의 예상  영향만을 고려한다 ....`
- 문단 BC105: `BC105 생산용식물에 IAS 16 의 재평가모형을 적용하여 회계처리하기로  선택한다면 가장 유의적인 영향은 재평가금액 ( 공정가치에 가깝다 )  의 변동을 기타포괄손익으로 인식하...`
- 문단 BC106: `BC106 개정 내용은 특정한 유형의 농림어업활동 ( 생산용식물을 보유한 기  업 ) 에만 영향을 미칠 것이다 ....`
- 문단 BC108: `BC108 IASB 는 다음과 같은 이유로 개정 내용이 기업 간 비교가능성을  유의적으로 떨어뜨릴 것으로 예상하지 않는다 .  ⑴ IAS 41 에서는 생물자산은 공정가치모형을 적용...`
- 문단 BC109: `BC109 IASB 는 개정 내용이 원가모형을 선택한 개별 기업의 보고기간 간  비교가능성을 유의적으로 떨어뜨릴 것으로 예상하지 않는다 . 이는  IAS 41 에 따른 생산용식물의...`
- 문단 BC110: `BC110 현재 생산용식물은 토지, 토지개량, 생산 과정에 사용된 농기계와  다르게 회계처리한다 . 대부분의 경우 이 자산들을 IAS 16 에 따라  원가로 회계처리한다 . 따라서...`
- 문단 BC111: `BC111 IAS 41 에서는 현재 생산용식물을 순공정가치로 측정하도록 하고  있다 . 따라서 공정가치 측정 규정은 생산용식물과 생산용식물에서  자라는 생산물 모두에 적용된다 . ...`
- 문단 BC112: `BC112 생산용식물의 생산물은 보통 판매하기 위해 재배한다 . 따라서 생  산물의 공정가치의 변동은 판매하여 받을 예상 미래현금흐름과  직접 관련이 있다 . 이와 달리 생산용식물...`
- 문단 BC113: `BC113 프로젝트를 진행하는 동안 스태프는 생산용식물을 보유한 회사의  재무제표를 이용하는 투자자와 재무분석가에게 의견을 구하였다 .  많은 투자자와 재무분석가는 기업이 실현할 ...`
- 문단 BC114: `BC114 개념체계에는 비슷한 자산은 비슷한 방식으로 회계처리하면 보고  정보의 유용성이 향상된다는 가정이 내재되어 있다 . 생산용식물은  공장 및 기계와 형태가 다르지만 사용하는...`
- 문단 BC115: `BC115 개정의 결과로 재무제표 이용자는 보통 생산용식물의 공정가치  정보 대신 원가 정보를 얻게 될 것이다 . 이것은 재무제표 이용자  들에게 덜 목적 적합한 정보를 제공하는 ...`
- 문단 BC116: `BC116 재무제표 작성자는 생산용식물의 활성시장이 없으면 특히 성숙도,  수확량, 입지가 다양한 큰 농장을 보유한 기업의 경우에도 공정가  치 측정이 복잡하고 시간이 많이 걸리며...`
- 문단 BC117: `BC117 그러나 개정 내용은 다음과 같은 이유로 대부분 기업의 준수 원  가를 줄여줄 것이다 .  ⑴ IASB 는 생산물을 순공정가치로 측정하는 것은 생산용식물과  생산물을 함께...`

---


검색 중: 유형자산 제거 시 손익 처리


### 쿼리: 유형자산 제거 시 손익 처리

#### [1] Parent: 제거
> `parent_id`: KIFRS1016_main_h_제거

**Matched Children:**

- 문단 **None** (score: 0.5260)  
  `유형자산의 장부금액은 다음과 같은 때에 제거한다  처분하는 때  사용이나 처분을 통하여 미래경제적효익이 기대되지 않을 때  유형자산의 제거로 생기는 손익은 자산을 제거할 때 당기손익으  으로 분류하지 않는다 [한]  68A 그러나 통상적인 활동과정에서 타인에게 임대할 목적으로 보유하  던 유형자산을 판매하는 기업은, 유형자산의 임대가 중단되고 판  매목적으...`

- 문단 **72** (score: 0.4872)  
  `72 유형자산의 제거에서 생기는 손익에 포함되는 대가 ( 금액 ) 는 기업  회계기준서 제 1115 호 문단 47~72 의 거래가격 산정에 관한 요구사  항에 따라 산정한다 . 손익에 포함된 추정 대가 ( 금액 ) 의 후속적인  변동은 기업회계기준서 제 1115 호의 거래가격 변동에 관한 요구사  항에 따라 회계처리한다 ....`

**Siblings (2건):**

- 문단 69: `69 유형자산은 여러 방법 ( 예 : 판매, 금융리스의 체결, 기부 ) 으로 처분  할 수 있다 . 유형자산의 처분일은 기업회계기준서 제 1115 호의 수  행의무 이행 시기를 판...`
- 문단 70: `70 문단 7 의 인식원칙에 따라 유형자산 항목의 일부에 대한 대체원  가를 자산의 장부금액으로 인식하는 경우, 대체되는 부분이 별도  로 분리되어 상각되었는지 여부와 관계없이 대...`

---

#### [2] Parent: 폐기와 처분
> `parent_id`: KIFRS1038_main_h_폐기와_처분

**Matched Children:**

- 문단 **None** (score: 0.5176)  
  `무형자산은 다음의 각 경우에 재무상태표에서 제거한다  처분하는 때  사용이나 처분으로부터 미래경제적효익이 기대되지 않을 때  무형자산의 제거로 생기는 이익이나 손실은 순매각금액과 장부금액  의 차이로 산정한다 그 이익이나 손실은 자산을 제거할 때 당기손  으로 분류하지 않는다                   - 42...`

**Siblings (4건):**

- 문단 114: `114 무형자산은 여러 방법 ( 예 : 매각, 금융리스의 체결, 기부 ) 으로 처분할  수 있다 . 무형자산의 처분일은 수령자가 기업회계기준서 제 1115 호  ‘ 고객과의 계약에...`
- 문단 115: `115 문단 21 의 인식원칙에 따라 무형자산의 일부에 대한 대체원가를 자  산의 장부금액으로 인식하는 경우, 대체된 부분의 장부금액은 제거  한다 . 대체된 부분의 장부금액을 실...`
- 문단 116: `116 무형자산의 제거에서 생기는 손익에 포함되는 대가 ( 금액 ) 는 기업회  계기준서 제 1115 호 문단 47~72 의 거래가격 산정에 관한 요구사항에  따라 산정한다 . 손...`
- 문단 117: `117 내용연수가 유한한 무형자산은 그 자산을 더 이상 사용하지 않을  때도 상각을 중지하지 아니한다 . 다만, 완전히 상각하거나 기업회계  기준서 제 1105 호에 따라 매각예정...`

---

#### [3] Parent: 매각예정으로 분류된 비유동자산 또는 처분자산집단 의 측정
> `parent_id`: KIFRS1105_main_h_매각예정으로_분류된_비유동자산_또는_처분자산집단_의_측정

**Matched Children:**

- 문단 **24** (score: 0.5014)  
  `24 비유동자산 ( 또는 처분자산집단 ) 의 매각일 전에 인식되지 않은 평  가손익은 재무상태표에서 제거되는 시점에 인식한다 . 제거와 관련  된 규정은 다음과 같다 .  ⑴ 유형자산의 경우 기업회계기준서 제 1016 호 ‘ 유형자산 ’ 의 문단  67∼72  ⑵ 무형자산의 경우 기업회계기준서 제 1038 호 ‘ 무형자산 ’ 의 문단  112∼117...`

**Siblings (15건):**

- 문단 None: `비유동자산 또는 처분자산집단 의 측정  매각예정으로 분류된 비유동자산 또는 처분자산집단 은 공정가치  에서 처분부대원가를 뺀 금액과 장부금액 중 작은 금액으로 측정  한다  소유주...`
- 문단 11: `11 참조 ) 을 충족한다면 문단 15 에 따라 최초 인식 시점에 공정가  치에서 처분부대원가를 뺀 금액과 매각예정으로 분류되지 않았을  경우의 장부금액 ( 예 : 원가 ) 중 작...`
- 문단 18: `18 자산 ( 또는 처분자산집단 ) 을 매각예정으로 최초 분류하기 직전에  해당 자산 ( 또는 처분자산집단 내의 모든 자산과 부채 ) 의 장부금액  은 적용가능한 한국채택국제회계기...`
- 문단 19: `19 그 이후 처분자산집단을 재측정하는 경우 매각예정으로 분류된  처분자산집단에 포함되지만 이 기준서의 측정 규정이 적용되지  않는 자산과 부채에 대해서는 적용가능한 한국채택국제회...`
- 문단 20: `20 자산 ( 또는 처분자산집단 ) 의 최초 또는 향후 공정가치에서 처분부  대원가를 뺀 금액의 하락을 손상차손으로 인식한다 . 다만, 문단...`
- 문단 21: `21 자산의 공정가치에서 처분부대원가를 뺀 금액이 증가하면 이익을  인식한다 . 그러나 그 금액은 이 기준서 또는 기업회계기준서 제...`
- 문단 22: `22 처분자산집단의 공정가치에서 처분부대원가를 뺀 금액이 증가하  면 이익으로 인식한다 . 이 경우 이익으로 인식하는 금액은 다음의  ⑴ 의 금액으로 하되, ⑵ 의 금액을 초과할 ...`
- 문단 23: `23 처분자산집단에 대하여 인식한 손상차손 ( 또는 손상차손환입 ) 은 기  업회계기준서 제 1036 호 ‘ 자산손상 ’ 의 문단 104 의 ⑴, ⑵ 및 문단...`
- 문단 25: `25 비유동자산이 매각예정으로 분류되거나 매각예정으로 분류된 처  분자산집단의 일부이면 그 자산은 감가상각 ( 또는 상각 ) 하지 아니  한다 . 매각예정으로 분류된 처분자산집단의...`
- 문단 26: `26 매각예정으로 분류되던 자산 ( 또는 처분자산집단 ) 이 문단 7∼9 의  요건을 더 이상 충족할 수 없다면 그 자산 ( 또는 처분자산집단 ) 은  매각예정으로 분류할 수 없으...`
- 문단 27: `27 더 이상 매각예정 또는 소유주에 대한 분배예정으로 분류할 수 없  거나 매각예정 또는 소유주에대한 분배예정으로 분류된 처분자산  집단에 포함될 수 없는 비유동자산 ( 또는 처...`
- 문단 28: `28 더 이상 매각예정 또는 소유주에 대한 분배예정으로 분류할 수  없는 비유동자산의 장부금액에반영하는 조정금액은 문단 7∼9 또는  5) 비유동자산이 현금창출단위의 일부라면 이 ...`
- 문단 29: `29 매각예정으로 분류된 처분자산집단에서 개별 자산이나 부채를 제  외하는 경우에 매각예정인 처분자산집단의 나머지 자산과 부채는  해당 집단이 문단 7∼9 의 요건을 충족해야만 계...`
- 문단 122: `122 에서 규정한 배분순서에 따라 집단에 속한 자산 중 이 기준서  의 측정 규정이 적용되는 비유동자산의 장부금액을 감소 ( 또는 증  가 ) 시킨다 ....`
- 문단 1036: `1036 호 ‘ 자산손상 ’ 에 따라 과거에 인식하였던 손상차손누계액을  초과할 수 없다 ....`

---

#### [4] Parent: 이 기준서의 주요 특징
> `parent_id`: KIFRS1016_bc_h_이_기준서의_주요_특징

**Matched Children:**

- 문단 **None** (score: 0.4945)  
  `이 기준서는 유형자산에 대한 회계처리와 공시에 대한 사항을 정하고 있다 .  적용범위  이 기준서는 적용범위에서 제외되는 일부 유형자산을 제외한 대부분의  유형자산에 대한 회계처리에 적용한다 . 다만, 다른 한국채택국제회계기준  서에서 상이한 회계처리를 요구하거나 허용하는 경우에는 이 기준서를  적용하지 아니한다 .  인식  유형자산으로 인식되기 위해서는 ...`

**Siblings:** 형제 없음 (단독 문단)

---



전체 3개 쿼리 검색 완료


## 4. 결과 요약

각 쿼리별 Parent 그룹 수, 매칭 Child 수, 형제 Child 수를 테이블로 정리합니다.

In [6]:
summary_lines = [
    "| 쿼리 | Parent 그룹 | 매칭 Child | 형제 Child (전체) | 평균 score |",
    "|---|---|---|---|---|",
]

for q, groups in all_results.items():
    n_groups = len(groups)
    n_matched = sum(len(g["matched_children"]) for g in groups)
    n_siblings = sum(len(g["siblings"]) for g in groups)
    scores = [mc["score"] for g in groups for mc in g["matched_children"]]
    avg_score = sum(scores) / len(scores) if scores else 0

    short_q = q if len(q) <= 25 else q[:22] + "..."
    summary_lines.append(
        f"| {short_q} | {n_groups} | {n_matched} | {n_siblings} | {avg_score:.4f} |"
    )

display(Markdown("\n".join(summary_lines)))

| 쿼리 | Parent 그룹 | 매칭 Child | 형제 Child (전체) | 평균 score |
|---|---|---|---|---|
| 유형자산 감가상각 방법 | 2 | 5 | 25 | 0.4979 |
| 원가모형과 재평가모형의 차이 | 4 | 5 | 37 | 0.4451 |
| 유형자산 제거 시 손익 처리 | 4 | 5 | 26 | 0.5053 |

## 5. 커스텀 쿼리 테스트

원하는 쿼리를 직접 입력하여 검색 결과를 확인할 수 있습니다.

In [7]:
# 쿼리를 변경하여 실행해 보세요
custom_query = "리스부채의 최초 측정"
custom_top_k = 5

print(f"커스텀 쿼리: {custom_query} (top_k={custom_top_k})")
custom_groups = search_with_parent(client, embeddings, custom_query, top_k=custom_top_k)
display_search_results(custom_query, custom_groups)

커스텀 쿼리: 리스부채의 최초 측정 (top_k=5)


### 쿼리: 리스부채의 최초 측정

#### [1] Parent: 리스이용자
> `parent_id`: KIFRS1116_main_h_리스이용자

**Matched Children:**

- 문단 **None** (score: 0.5749)  
  `인식  리스이용자는 리스개시일에 사용권자산과 리스부채를 인식한다  측정  최초 측정  사용권자산의 최초 측정  리스이용자는 리스개시일에 사용권자산을 원가로 측정한다...`

- 문단 **1002** (score: 0.5426)  
  `1002 호를 적용하여 회계처리하는 그러한 원가에 대한 의무는 기  업회계기준서 제 1037 호 ‘ 충당부채, 우발부채, 우발자산 ’ 을 적용하  여 인식하고 측정한다 .  리스부채의 최초 측정  리스이용자는 리스개시일에 그날 현재 지급되지 않은 리스료의  현재가치로 리스부채를 측정한다 리스의 내재이자율을 쉽게 산정  할 수 있는 경우에는 그 이자율로 리스...`

**Siblings (33건):**

- 문단 6: `6 월 30 일 후에 리스료를 증가시키는 경우 이러한 조건을 만족  시킨다 ).  ⑶ 그 밖의 리스기간과 조건은 실질적으로 변경되지 않는다 .  표시...`
- 문단 24: `24 사용권자산의 원가는 다음 항목으로 구성된다 .  ⑴ 리스부채의 최초 측정금액 ( 문단 26 에서 기술함 )  ⑵ 리스개시일이나 그 전에 지급한 리스료 ( 받은 리스 인센티브는...`
- 문단 25: `25 리스이용자는 문단 24⑷ 에서 기술하는 원가에 대한 의무를 부담  할 때 사용권자산 원가의 일부로 그 원가를 인식한다 . 리스이용자  는, 특정한 기간에 재고자산을 생산하기 ...`
- 문단 27: `27 리스개시일에 리스부채의 측정치에 포함되는 리스료는, 리스기간  에 걸쳐 기초자산을 사용하는 권리에 대한 지급액 중 그날 현재  지급되지 않은 다음 금액으로 구성된다 .  ⑴ ...`
- 문단 28: `28 문단 27⑵ 에서 기술하는 지수나 요율 ( 이율 ) 에 따라 달라지는 변동  리스료의 예에는 소비자물가지수에 연동되는 지급액, 기준금리 ( 예 :  LIBOR) 에 연동되는 ...`
- 문단 30: `30 원가모형을 적용하기 위하여, 리스이용자는 원가에서 다음을 차감  하고 조정하여 사용권자산을 측정한다 .  ⑴ 감가상각누계액과 손상차손누계액을 차감  ⑵ 문단 36⑶ 에서 규정...`
- 문단 31: `31 리스이용자는 사용권자산을 감가상각할 때 문단 32 의 요구사항을  전제로 기업회계기준서 제 1016 호 ‘ 유형자산 ’ 의 감가상각에 대한  요구사항을 적용한다 ....`
- 문단 32: `32 리스가 리스기간 종료시점 이전에 리스이용자에게 기초자산의 소  유권을 이전하는 경우나 사용권자산의 원가에 리스이용자가 매수  선택권을 행사할 것임이 반영되는 경우에, 리스이용...`
- 문단 33: `33 리스이용자는 사용권자산이 손상되었는지를 판단하고 식별되는  손상차손을 회계처리하기 위하여 기업회계기준서 제 1036 호 ‘ 자산  손상 ’ 을 적용한다 .  다른 측정모형...`
- 문단 34: `34 리스이용자가 투자부동산에 기업회계기준서 제 1040 호 ‘ 투자부동  산 ’ 의 공정가치모형을 적용하는 경우에는, 기업회계기준서 제 1040  호의 투자부동산 정의를 충족하는...`
- 문단 35: `35 사용권자산이 기업회계기준서 제 1016 호의 재평가모형을 적용하는  유형자산의 유형에 관련되는 경우에, 리스이용자는 그 유형자산의  유형에 관련되는 모든 사용권자산에 재평가모...`
- 문단 37: `37 리스기간 중 각 기간의 리스부채에 대한 이자는 리스부채 잔액에  대하여 일정한 기간이자율이 산출되도록 하는 금액이다 . 기간이자  율은 문단 26 에서 기술하는 할인율이거나,...`
- 문단 38: `38 리스이용자는 리스개시일 후에 다음 원가를 모두 당기손익으로  인식한다 . 다만, 적용 가능한 다른 기준서를 적용하는, 다른 자산  의 장부금액에 포함되는 원가인 경우는 제외한...`
- 문단 39: `39 리스이용자는 리스개시일 후에 리스료에 생기는 변동을 반영하기  위하여 리스부채를 다시 측정할 때 문단 40~43 을 적용한다 . 리스  이용자는 사용권자산을 조정하여 리스부채...`
- 문단 40: `40 리스이용자는 다음 중 어느 하나에 해당하는 경우에 수정 할인율  로 수정 리스료를 할인하여 리스부채를 다시 측정한다 .  ⑴ 리스기간에 변경이 있는 경우 ( 문단 20~21 ...`
- 문단 41: `41 문단 40 을 적용할 때, 리스이용자는 내재이자율을 쉽게 산정할 수  있는 경우에는 남은 리스기간의 내재이자율로 수정 할인율을 산  정하나, 리스의 내재이자율을 쉽게 산정할 ...`
- 문단 42: `42 리스이용자는 다음 중 어느 하나에 해당하는 경우에 수정 리스료  를 할인하여 리스부채를 다시 측정한다 .  ⑴ 잔존가치보증에 따라 지급할 것으로 예상되는 금액에 변동이  있는...`
- 문단 43: `43 변동이자율의 변동으로 리스료에 변동이 생긴 것이 아니라면 문  단 42 를 적용할 때 리스이용자는 변경되지 않은 할인율을 사용한                      - 19...`
- 문단 44: `44 리스이용자는 다음 조건을 모두 충족하는 리스변경을 별도 리스  로 회계처리한다 .  ⑴ 하나 이상의 기초자산 사용권이 추가되어 리스의 범위가 넓어  진다 .  ⑵ 넓어진 리스...`
- 문단 45: `45 별도 리스로 회계처리하지 않는 리스변경에 대하여 리스이용자는  리스변경 유효일에 다음과 같이 처리한다 .  ⑴ 문단 13~16 을 적용하여 변경된 계약의 대가를 배분한다 . ...`
- 문단 46: `46 별도 리스로 회계처리하지 않는 리스변경에 대하여 리스이용자는  다음과 같이 리스부채의 재측정을 회계처리한다 .  ⑴ 리스의 범위를 좁히는 리스변경에 대하여 리스의 일부나 전부...`
- 문단 47: `47 리스이용자는 다음과 같이 재무상태표에 표시하거나 주석으로 공  시한다 .  ⑴ 사용권자산을 다른 자산과 구분하여 표시하거나 공시한다 . 리  스이용자가 재무상태표에서 사용권자...`
- 문단 48: `48 문단 47⑴ 의 요구사항은 투자부동산의 정의를 충족하는 사용권자  산에는 적용하지 않고, 그 사용권자산은 재무상태표에 투자부동산  으로 표시한다 ....`
- 문단 49: `49 포괄손익계산서에서 리스이용자는 리스부채에 대한 이자비용을  사용권자산의 감가상각비와 구분하여 표시한다 . 리스부채에 대한  이자비용은 기업회계기준서 제 1001 호 ‘ 재무제...`
- 문단 50: `50 리스이용자는 현금흐름표에서 다음과 같이 분류한다 .  ⑴ 리스부채의 원금에 해당하는 현금 지급액은 재무활동으로 분  류  ⑵ 리스부채의 이자에 해당하는 현금 지급액은 기업회계...`
- 문단 52: `52 리스이용자는 재무제표에서 하나의 주석이나 별도로 구분되는 난  (section) 으로 리스이용자의 리스에 대한 정보를 공시한다 . 그러나  리스이용자는 리스에 대한 정보가 하...`
- 문단 53: `53 에서 규정하는 내용을 공시한다 . 공시하는 금액에는 리스이용자  가 보고기간 중에 다른 자산의 장부금액에 포함한 원가를 포함한  다 ....`
- 문단 55: `55 보고기간 말 현재 약정된 단기리스 포트폴리오가 문단 53⑶ 을 적  용하여 공시하는 단기리스 비용에 관련되는 단기리스 포트폴리오  와 다른 경우에, 리스이용자는 문단 6 을 ...`
- 문단 56: `56 사용권자산이 투자부동산의 정의를 충족한다면, 리스이용자는 기  업회계기준서 제 1040 호의 공시 요구사항을 적용한다 . 이 경우에  리스이용자는 해당 사용권자산에 대하여 문...`
- 문단 57: `57 리스이용자가 기업회계기준서 제 1016 호를 적용하여 사용권자산을  재평가금액으로 측정하는 경우에, 해당 사용권자산에 대하여 기업  회계기준서 제 1016 호 문단 77 에서...`
- 문단 58: `58 리스이용자는 다른 금융부채의 만기분석과는 별도로 기업회계기  준서 제 1107 호 ‘ 금융상품 : 공시 ’ 의 문단 39 와 B11 을 적용하여 리스  부채의 만기분석 내용을...`
- 문단 59: `59 문단 53~58 에서 요구하는 공시에 추가하여, 리스이용자는 문단 51  의 공시 목적을 이루기 위하여 필요한, 리스 활동에 대한 추가 질  적   - 양적 정보를 공시한다 ...`
- 문단 60: `60 단기리스나 소액자산 리스에 문단 6 을 적용하는 리스이용자는 그  사실을 공시한다 .  60A 문단 46A 의 실무적 간편법을 적용하는 리스이용자는 다음 사항을  공시한다 ....`

---

#### [2] Parent: 개정
> `parent_id`: KIFRS1012_bc_h_개정

**Matched Children:**

- 문단 **BC90** (score: 0.5525)  
  `BC90 IFRS 16 을 적용하면, 리스개시일에 그날 현재 지급되지 않은 리스  료의 현재가치로 리스부채를 처음 측정한다 . 리스자산의 최초 측정  치에는 리스부채의 최초 측정금액뿐만 아니라 선급리스료와 리스  개설직접원가도 포함된다 ....`

**Siblings (45건):**

- 문단 BC71: `BC71 2021 년 5 월에 IASB 는 ‘ 단일 거래에서 생기는 자산과 부채에 관련  되는 이연법인세 ’ 를 공표하였다 . 이 개정 내용은 IAS 12 문단 15  와 24 의...`
- 문단 BC72: `BC72 이 개정 내용은 해석위원회의 권고에 대응하여 공표되었다 . 해석  위원회가 수행한 조사에 따르면, 리스와 같이 자산과 부채를 인식  하게 하는 거래에 인식 예외규정이 적용...`
- 문단 BC73: `BC73 문단 BC74~BC91 에서는 단순하게 리스를 예로 사용하여 개정 내  용의 근거를 설명한다 . 이 설명은 이 개정 내용의 영향을 받는 다  른 거래 [ 예 : 사후처리 ...`
- 문단 BC74: `BC74 IFRS 16 을 적용하면, 리스개시일에 사용권자산 ( 이하 ‘ 리스자산 ’ 이라  한다 ) 과 리스부채를 인식한다 . 리스자산과 리스부채를 처음 인식하  는 시점에, 이...`
- 문단 BC75: `BC75 세무상 공제가 리스자산과 리스부채 중 어느 항목에 귀속되는지는  적용되는 세법을 고려하여 판단한다 .                   - 133...`
- 문단 BC76: `BC76 IAS 12 를 적용하면, 최초 인식시점에 일시적차이는 기업이 세무상  공제가 리스부채에 귀속된다고 판단하는 경우에만 생긴다 . 그 이유  는 다음과 같다 .  ⑴ 세무상...`
- 문단 BC77: `BC77 이 개정 내용이 공표되기 전에는 문단 BC76⑵ 에 기술된 상황에서 생  기는 일시적차이에 인식예외규정을 적용할지에 대하여 서로 다른  견해가 존재하였다 . 기업이 인식예...`
- 문단 BC78: `BC78 IAS 12 문단 22⑶ 에서는 인식예외규정의 목적을 설명한다 . 사업결  합이 아니고 회계이익과 과세소득에 영향을 미치지 않는 거래에서  자산이나 부채를 최초로 인식할 ...`
- 문단 BC79: `BC79 에서 설명하는 바와 같이, 동일한 금액으로 가산할 일시적차  이와 차감할 일시적차이가 생길 수 있다 . 이 개정 내용은 그렇게  생기는 동일한 금액의 가산할 일시적차이와 ...`
- 문단 BC80: `BC80 문단 BC79 에 요약된 의견을 고려하여, IASB 는 최초 인식시점에  동일한 금액으로 가산할 일시적차이와 차감할 일시적차이를 생기  게 하는 거래에 인식예외규정이 적용...`
- 문단 BC81: `BC81 IASB 는 동일한 금액의 가산할 일시적차이와 차감할 일시적차이에  대하여 같은 금액으로 이연법인세 자산 및 부채를 인식하지 않을  때, 이 축소된 적용범위의 인식 예외규...`
- 문단 BC82: `BC82 IAS 12 문단 24 에서는 ‘ 차감할 일시적차이가 사용될 수 있는 과세  소득의 발생 가능성이 높은 경우 ’ 에만 이연법인세자산을 인식하도  록 요구한다 ( 이하 ‘ ...`
- 문단 BC83: `BC83 이러한 상황을 해결하기 위하여 IASB 는 외부검토의견을 수렴하기  위한 개정 공개초안을 공개하였을 때, 인식 예외규정을 적용하지  않으면 생길 이연법인세 자산 및 부채의...`
- 문단 BC84: `BC84 개정 공개초안에 대한 외부검토의견은 다음과 같은 사실을 나타낸  다 .  ⑴ 상한 제안은 IAS 12 의 원칙과 일관되지 않을 것이다 . 이 기준  서에서는 모든 가산할 ...`
- 문단 BC85: `BC85 이러한 외부검토의견에 대응하여, IASB 는 상한 제안을 삭제하였고,  다음과 같이 결론 내렸다 .                   - 136  ⑴ 회수 가능성 요건이 적...`
- 문단 BC86: `BC86 상한 제안을 삭제하면 거래의 최초 인식시점에 동일하지 않은 금  액으로 이연법인세 자산 및 부채를 인식하는 결과를 가져올 수 있  다 . 그러한 경우, 그 차액을 당기손익...`
- 문단 BC87: `BC87 더욱이, IASB 는 같은 거래에서 생기는 가산할 일시적차이의 미래  소멸을 통하여 회수 가능성 요건을 보통 충족할 수 있을 것이기  때문에, 최초 인식시점에 동일하지 않...`
- 문단 BC88: `BC88 이연법인세 자산 및 부채의 측정치에 서로 다른 세율이 적용되는  경우에는 동일한 금액의 가산할 일시적차이와 차감할 일시적차이  에 대하여 서로 다른 금액으로 이연법인세 자...`
- 문단 BC89: `BC89 개정 공개초안에 대한 일부 의견제출자는 세무상 공제가 리스자산  과 리스부채 중 어느 항목에 귀속되는지를 평가하는 데 도움을 주  는 적용지침을 IASB 가 제공해 달라고...`
- 문단 BC92: `BC92 IASB 는 다음과 같은 이유로 개정 내용의 예상 효익이 원가를 초과  한다고 결론 내렸다 .  ⑴ 개정 내용은 리스 및 사후처리 의무와 같은 거래의 보고 다양  성을 줄...`
- 문단 BC93: `BC93 IASB 는 IAS 8 에 따라 개정 내용을 소급 적용하도록 요구하지는 않  기로 결정하였다 . 그 대신에 비교 표시되는 가장 이른 기간의 시작  일에 리스 및 사후처리 ...`
- 문단 BC94: `BC94 IASB 는 리스 및 사후처리 의무 외의 거래 ( 비교 표시되는 가장 이  른 기간의 시작일 이후에 이루어지는 그러한 거래 ) 에도 이 개정  내용을 전진적으로 적용하도록...`
- 문단 BC95: `BC95 문단 BC93 에서 설명하는 바와 비슷한 이유로, IASB 는 최초채택기  업이 IFRS 전환일에 존재하는 리스 및 사후처리 의무에 관련되는  모든 일시적차이에 대하여 이...`
- 문단 BC96: `BC96 2023 년 5 월에 IASB 는 ‘ 국제조세개혁 - 필라 2 모범규칙 ’ 을 공표하  였고 개정 내용에 다음과 같은 규정을 도입하였다 .  ⑴ 필라 2 법인세와 관련되는...`
- 문단 BC97: `BC97 2021 년 10 월, 135 개국 이상의 국가가 경제협력개발기구  (OECD)/G20 의 세원잠식과 소득이전에 대한 포괄적 이행체계가  발표한 ‘ 경제의 디지털화에서 생...`
- 문단 BC98: `BC98 필라 2 모범규칙은 합의된 공통 접근법의 일부로서 각국이 국내  법인세법으로 전환하여 시행할 수 있는 틀을 제공한다 . 이 규칙은  다음과 같다 .  ⑴ 대규모 다국적 연...`
- 문단 BC99: `BC99 이해관계자들은 단시일 내에 필라 2 모범규칙을 시행하는 국가에서  생기는 법인세 회계처리의 영향에 대한 우려를 IASB 에 알렸다 . 그  우려는 다음과 관련된다 .  ⑴...`
- 문단 BC100: `BC100 이해관계자들의 우려를 고려하여 IASB 는 기업이 추가세액과 관련  되는 이연법인세를 회계처리하기 위하여 IAS 12 의 원칙과 요구사  항을 적용하는 방법을 판단하는 ...`
- 문단 BC101: `BC101 그러므로 IASB 는 필라 2 법인세와 관련하여, 이연법인세 자산 및  부채를 인식하고 이에 대한 정보를 공시하도록 하는 IAS 12 의 요  구사항에 대한 한시적 예외...`
- 문단 BC102: `BC102 또 IASB 는 기업이 한시적 예외 규정을 적용하였다는 사실을 공시  하도록 요구하기로 결정하였다 . IASB 는 이 요구사항이 다음과 같  은 결과를 가져올 것이라고 ...`
- 문단 BC103: `BC103 IASB 는 추가세액이 법인세인 상황에 대한 추가 설명이나 지침을  제공하지 않기로 결정하였다 ( 문단 BC99⑴ 참조 ). IASB 는 긴급히  필요한 개정의 완료를 ...`
- 문단 BC104: `BC104 IASB 는 한시적 예외 규정의 적용범위를 국내 법인세법에 따라 인  식된 이연법인세의 측정을 포함하도록 확대할 필요는 없다고 결  론 내렸다 . IASB 는 기업이 관...`
- 문단 BC105: `BC105 IASB 는 다음과 같은 이유로 한시적예외 규정을 의무 적용하도록  결정하였다 .  ⑴ 재무제표의 기업 간 비교 가능성이 높아져 재무제표이용자에게  더 유용한 정보를 제...`
- 문단 BC106: `BC106 IASB 는 문단 BC100 에서 기술한 활동에 필요한 시간을 산정할 수  없다고 결론 내렸다 . 그 활동들은 국가들이 필라 2 모범규칙을 어               ...`
- 문단 BC107: `BC107 필라 2 법률이 제정되었거나 실질적으로 제정되었지만 아직 시행  일이 도래하지는 않은 기간에 재무제표이용자는 그 법률에서 생  기는 필라 2 법인세에 대한 기업의 익스포...`
- 문단 BC108: `BC108 IASB 는 필라 2 법률이 제정되었거나 실질적으로 제정되었지만 아  직 시행일이 도래하지 않은 기간에기업은 필라 2 법률을 따르기  위한 준비와 기업의 익스포저를 평가...`
- 문단 BC109: `BC109 검토의견을 고려하는 과정에서, IASB 는 일부 국가에서 2024 년 1  월 1 일부터 그 법률이 시행될 예정임을 알게 되었다 . 그러므로  IASB 는 공시 요구사항...`
- 문단 BC110: `BC110 문단 BC108~BC109 에서 논의된 요소들의 균형을 맞추기 위해  IASB 는 다음과 같이 하기로 결정하였다 .  ⑴ 공시 목적을 이루는 정보를 공시하도록 기업에 요...`
- 문단 BC111: `BC111 IASB 위원 일부는 기업이 알고 있거나 합리적으로 추정할 수 있  는 정보만 공시하도록 요구하면 일부 기업이 공시 목적을 이루기  위한 양적 정보를 공시하지 않는 결과...`
- 문단 BC112: `BC112 또 IASB 는 공시 목적을 이루기 위해 기업에 다음과 같은 공시를  요구하기로 결정하였다 .  ⑴ 양적 특성과 질적 특성을 가진 정보를 모두 공시해야 한다 .  IAS...`
- 문단 BC113: `BC113 더욱이 IASB 는, 기업이 공시 목적을 이루기 위해 공시해야 하는  정보는 필라 2 법률의 구체적인 요구사항을 모두 반영할 필요는  없고 대략적인 범위의 형태로 제공될...`
- 문단 BC114: `BC114 IASB 는 필라 2 법인세와 관련되는 당기법인세비용을 별도로 공시  하도록 요구하기로 결정했다 . IASB 는 이 정보가 다음과 같을 것  이라고 결론 내렸다 .  ⑴...`
- 문단 BC115: `BC115 IASB 는 다음과 같은 이유로 이 개정의 효익이 원가를 초과한다  고 결론 내렸다 .  ⑴ 개정 내용은 영향을 받는 기업에 적시에 면제 규정을 제공하여  실무에서 IA...`
- 문단 BC116: `BC116 IASB 는 다음과 같이 결론 내렸다 .  ⑴ 한시적 예외 규정이 효과적이기 위해서는 개정 내용이 공표되  는 즉시 기업이 그 규정을 적용할 수 있어야 한다 .  ⑵ 문...`
- 문단 BC117: `BC117 IASB 는 기업이 한시적 예외 규정을 소급 적용하도록 요구하기로  결정하였다 . 이 요구사항은 필라 2 법률이 제정되었거나 실질적으  로 제정된 날 ( 비록 그 날이 ...`

---

#### [3] Parent: 중요성
> `parent_id`: KIFRS1116_bc_h_중요성

**Matched Children:**

- 문단 **BC143** (score: 0.5374)  
  `BC143 IASB 는 사용권자산과 리스부채의 최초 측정과 관련하여 측정은  리스 거래의 특성과 조건을 반영한다는 점에 주목하였다 . 이에 따  라 리스이용자는 약정일 ( 개시일 전일 수 있음 ) 에 계약상 합의된 조  건들을 주목해야 할 것이다 . 그러나 약정일을 최초 측정일로 본다  면 개시일에 리스자산과 리스부채를 인식할 때 약정일과 개시일  사이의 ...`

**Siblings (50건):**

- 문단 BC84: `BC84 많은 리스이용자가 개수는 많으나 가치가 낮은 리스 ( 특히 그 리스  의 총 가치가 재무제표 전체적으로 미치는 영향이 적은 경우 ) 에  IFRS 16 의 요구사항을 적용...`
- 문단 BC85: `BC85 이 우려를 고려하여, IASB 는 IFRS 16 에 중요성에 대한 분명한 지  침을 포함하는 것 ( 사소한 리스는 IFRS 16 의 적용범위에서 제외된           ...`
- 문단 BC86: `BC86 IFRS 16 에 중요성 지침을 포함하지 않기로 결정하는 과정에서,  IASB 는 IFRS 16 의 인식 및 측정 요구사항을 적용하는 영향이 재  무제표에서 중요하지 않다...`
- 문단 BC87: `BC87 IASB 는 리스이용자의 단기리스에 IFRS 16 의 모든 요구사항을 적  용하는 효익이 관련되는 원가를 초과하지 않는다고 결론 내렸다 .  리스이용자의 원가를 줄이는 방...`
- 문단 BC88: `BC88 IASB 는 단기리스에 대한 측정 요구사항 단순화를 고려하였다 . 구  체적으로는 단기리스에서 생기는 자산 및 부채를 측정하기 위하여  사용되는 지급액을 할인하게 하는 요...`
- 문단 BC89: `BC89 IASB 는 비록 단순화된 측정 요구사항이 있을지라도 리스이용자에  게 단기리스의 사용권자산과 리스부채를 인식하도록 요구한다면  그 효익이 관련되는 원가를 초과하지는 않을...`
- 문단 BC90: `BC90 단기리스에 대한 면제 규정이 소액자산 리스에는 충분한 경감 규  정이 되지 못한다는 의견을 고려하여, IASB 는 그 리스에 대한 별  도 면제 규정 ( 문단 BC98∼B...`
- 문단 BC91: `BC91 IASB 는 처음에, 단기리스를 개시일에 가능한 최대 리스기간이 12  개월 이하인 리스로 정의하는 것을 고려하였다 . 그러나 많은 이해  관계자는 이 방식으로 단기리스의...`
- 문단 BC92: `BC92 이 의견을 고려하여, IASB 는 12 개월을 초과하는 리스로 단기리스  면제 규정을 확장하는 것을 고려하였다 . 일부 이해관계자는 ‘ 단기 ’  가 5 년까지 되어야 한...`
- 문단 BC93: `BC93 그 대신에, IASB 는 리스기간의 산정과 일치하도록 연장선택권의  행사 가능성과 종료선택권의 미행사 가능성을 고려하여 단기리스  의 만기를 산정함으로써 단기리스 면제 규...`
- 문단 BC94: `BC94 이 결론에 이르는 과정에서, IASB 는 단기리스 면제 규정을 충족하  도록 리스를 구조화할 수 있는 위험을 고려하였다 . IASB 는 리스제  공자에게 미치는 단기리스의...`
- 문단 BC95: `BC95 IASB 는 가능한 최대 기간 대신에 IFRS 16 의 리스기간 산정을 참  조하여 단기리스를 정의함으로써 잃을 수 있는 증분 정보는 적을  것이라고 보았다 . 이는 리스...`
- 문단 BC96: `BC96 IASB 는 IFRS 16 의 리스기간 산정을 사용하여 단기리스를 식별하  는 방식이 적용하기에 더 복잡할지도 고려하였다 . 리스기간을 식  별하기 위해 최대 기간보다 판...`
- 문단 BC98: `BC98 문단 BC84 에서 본 바와 같이, 많은 리스이용자가 개수는 많으나  가치가 낮은 리스에 IFRS 16 의 요구사항을 적용하는 원가에 대한  우려를 나타냈다 . 그들은 그...`
- 문단 BC99: `BC99 이 우려를 고려하여, IASB 는 소액자산 리스에 대하여 인식 면제  규정을 제공하기로 결정하였다 . 따라서 IFRS 16 에서는 리스별로  기초자산이 소액인 경우에 그 ...`
- 문단 BC101: `BC101 IASB 는 사용권자산과 리스부채가 리스이용자의 재무제표에 인식  되는 경우에 소액자산 리스가 미칠 영향을 평가하는 현장연구  (fieldwork) 를 수행하였다 . 그...`
- 문단 BC102: `BC102 IASB 는 어떤 경우에는 면제 규정에 해당하는 리스 총액의 가치가  중요할 수도 있다는 위험을 인정하였다 . IASB 의 현장연구 결과는  여러 개의 소액자산 개별 리...`
- 문단 BC103: `BC103 IASB 는 소액자산 리스에 대한 인식 면제 규정을 리스별로 적용해  야 한다고 결정하였다 . 리스별이 아닌 기초자산의 유형별로 적용  하도록 요구한다면 리스이용자에게 ...`
- 문단 BC104: `BC104 IASB 는 또 리스이용자가 인식 면제 규정을 적용하기로 선택하는  경우에 리스이용자에게 소액자산 리스에 관련하여 인식된 비용 금  액을 공시하도록 요구하기로 결정하였다...`
- 문단 BC105: `BC105 IFRS 16 에서는 고객이 식별되는 자산의 사용을 일정 기간 통제하  는지에 기초하여 리스를 정의한다 . 고객이 식별되는 자산의 사용  을 일정 기간 통제한다면 그 계...`
- 문단 BC106: `BC106 2010 년 공개초안에서는, IAS 17 의 리스 정의와 IFRIC 4 의 부수적  인 요구사항을 근본적으로 유지하였다 . 많은 의견제출자는 제안된  요구사항에 포착되는...`
- 문단 BC107: `BC107 따라서 IASB 는 그 우려를 해결하기 위하여 2013 년 공개초안에서  리스의 정의에 대한 지침 변경을 제안하였다 . 2013 년 공개초안에서  는 용역과 리스를 구별...`
- 문단 BC109: `BC109 IASB 의 견해는 대부분의 경우에 계약이 리스를 포함하는지를 판  단하기는 쉬울 것이라는 것이다 . 계약은 많은 요구사항을 충족하  지 못하여 리스의 정의를 충족하지 ...`
- 문단 BC113: `BC113 IASB 는 대체권이 실질적인 상황을 판단하는 데 도움이 되는 적용  지침을 포함하였다 . 이 지침은 공급자가 자산을 대체할 실질적인  능력이 있고 대체하여 경제적으로 ...`
- 문단 BC114: `BC114 대체권은 몇 가지 이유로 실질적이지 않을 수 있다 . 공급자가 그  자산을 대체할 수 있는 경우를 계약에서 제한하기 때문에 일부 대  체권은 실질적이지 않다 . 예를 들...`
- 문단 BC115: `BC115 이해관계자들은 경우에 따라 공급자의 대체권이 실질적인지를 고  객이 판단하기가 불가능하지는 않지만 어려울 수 있다는 우려를  제기하였다 . 이 어려움은 고객이 흔히 공급...`
- 문단 BC116: `BC116 IFRS 16 에서는 식별되는 자산이 되려면 자산이 물리적으로 구별되  어야 한다고도 명확히 한다 . IASB 는 더 큰 자산의 용량 (capacity)  일부가 물리적...`
- 문단 BC120: `BC120 IASB 의 견해는 자산을 사용하는 방법 및 목적에 대한 결정이 자  산의 사용에 대하여 내리는 다른 결정 ( 자산의 운용 및 유지에 대  한 결정을 포함함 ) 보다 자...`
- 문단 BC121: `BC121 IASB 는 자산을 사용하는 방법 및 목적에 대한 결정이 미리 내려  져서 사용기간 중에 고객이나 공급자가 그 결정을 내릴 수 없는  경우가 있다는 점에 주목하였다 . ...`
- 문단 BC122: `BC122 자산을 사용하는 방법 및 목적에 대한 결정이 미리 내려지는 경우  에는 고객에게 식별되는 자산의 사용을 지시할 권리가 있는지를  판단하는 접근법이 달라진다 . IFRS ...`
- 문단 BC124: `BC124 또 IFRS 16 에서는 방어권 ( 예 : 기초자산 또는 그 밖의 자산에 대한  공급자의 지분이나 공급자의 인력을 보호하기 위해서나, 공급자가  관련 법규를 지킬 수 있...`
- 문단 BC125: `BC125 IASB 는 IFRS 16 을 개발하면서 리스의 정의에 대하여 이해관계자  들이 제안한 대안들을 고려하였다 . 고려하였던 주요 대안은 다음  과 같다 .  ⑴ 금융요소 ...`
- 문단 BC126: `BC126 둘 이상의 당사자가 IFRS 11 ‘ 공동약정 ’ 에서 정의하는 공동지배력  을 가지는 공동약정을 구성할 때 그 당사자는 공동약정의 운영에  사용할 자산을 리스하기로 결...`
- 문단 BC127: `BC127 IFRS 16 의 적용범위를 정하기 위해 IASB 는 계약에서 집행 가능한  권리와 의무가 생기는 경우에만 계약이 존재한다고 볼 것이라고  결정하였다 . 리스에서 모든 ...`
- 문단 BC128: `BC128 따라서 리스이용자가 리스를 연장하거나 종료할 권리를 가진다면                     - 204  최초 해지불능기간을 초과하는 집행 가능한 권리와 의무가 있고...`
- 문단 BC129: `BC129 IASB 는 이 방식으로 리스에 집행 가능성을 적용하면 경제적 실질  이 없는 리스 조항 ( 예 : 리스가 실무적으로는 해지되지 않을 것을  알면서도 언제라도 해지할 수...`
- 문단 BC130: `BC130 IASB 는 보통 계약을 개별적으로 회계처리하는 것이 적절하지만,  상호의존적인 계약들은 결합된 영향을 평가하는 것도 필요하다는                     - ...`
- 문단 BC131: `BC131 IASB 는 계약을 결합해야 하는 상황을 식별하는 데 ‘ 개념체계 ’ 의  표현 충실성이라는 개념으로 충분하다는 일부 견해가 있다는 점에  주목하였다 . 그러나 IASB...`
- 문단 BC132: `BC132 따라서 IASB 는 계약들을 단일 계약으로 결합하여 회계처리해야  하는 상황을 IFRS 16 에서 규정하기로 결정하였다 . 그 요구사항은  IFRS 15 의 요구사항과 ...`
- 문단 BC133: `BC133 일부 계약은 리스요소와 비리스 ( 용역 ) 요소를 모두 포함한다 . 예를  들면 차량에 대한 계약은 리스와 유지용역을 결합할 수 있다 . 또  많은 계약은 둘 이상의 리...`
- 문단 BC134: `BC134 IFRS 16 에서는 리스를 포함하는 계약에 오직 하나의 리스요소만  있는지, 몇 가지 리스요소가 있는지를 판단하기 위한 요구사항을  포함한다 . IASB 는 리스계약에...`
- 문단 BC136: `BC136 IFRS 16 에서는 거래가격을 수행의무에 배분하게 하는 IFRS 15 의  요구사항을 적용하여 리스제공자가 계약의 대가를 리스요소와 비  리스요소에 배분하도록 요구한다...`
- 문단 BC140: `BC140 IFRS 16 은 기초자산 사용권을 일정 기간 이전하는 계약에 적용되  고 기업에 기초자산에 대한 통제를 이전하는 거래에는 적용되지  않는다 . 그러한 거래는 다른 기준...`
- 문단 BC144: `BC144 IASB 는 이 접근법이 다음과 같은 효익이 있다는 점에 주목하였다 .  ⑴ 리스이용자가 사용권자산과 리스부채를 처음으로 인식할 때 차  손익이 생기지 않아야 한다는 점...`
- 문단 BC145: `BC145 IASB 는 리스료의 현재가치를 참조하여 측정한 원가를 기준으로  사용권자산과 리스부채를 측정 ( 원가 측정기준 ) 하도록 요구하기로  결정하였다 . IASB 는 이 접...`
- 문단 BC146: `BC146 IASB 는 사용권자산과 리스부채의 최초 및 후속 측정을 IFRS 16 에  서 규정하지 않고 다른 기준서를 참조할지를 고려하였다 . IASB 는  다음과 같은 이유로 ...`
- 문단 BC148: `BC148 IASB 는 리스이용자가 기초자산의 사용으로 생기는 경제적 효익에  대해 더 목적 적합한 정보를 제공할 수 있는 공정가치로 사용권자  산을 처음 측정해야 하는지를 고려하...`
- 문단 BC149: `BC149 IFRS 16 에서는 리스이용자가 사용권자산을 처음 측정할 때 리스개  설직접원가를 사용권자산 원가에 포함하고 리스기간에 걸쳐 그 원  가를 상각하도록 요구한다 . 사용...`
- 문단 BC150: `BC150 IASB 는 리스이용자와 리스제공자가 같은 리스개설직접원가의 정  의를 적용해야 한다고 결정하였다 . 이 결정은 주로 IFRS 16 을 적  용하는 과정에서 복잡성을 줄...`
- 문단 BC151: `BC151 IASB 는 개시일에 리스이용자가 부담하는 리스개설직접원가를 사  용권자산과 리스부채에 배분해야 하는지를 고려하였다 . 그러나  IASB 는 기업들이 그러한 접근법을 적...`

---

#### [4] Parent: 사례 리스
> `parent_id`: KIFRS1012_ie_h_사례_리스

**Matched Children:**

- 문단 **None** (score: 0.5280)  
  `리스  기업 ( 리스이용자 ) 은 건물을 5 년 동안 리스하는 계약을 체결하였다 . 연간 리스  료는 100 원이고 매년 말에 지급한다 . 리스개시일 전에 리스이용자는 리스료  15 원을 지급하고 ( 선급리스료 ) 리스개설직접원가 5 원을 지급한다 . 리스의 내재  이자율은 쉽게 산정할 수 없다 . 리스이용자의 증분차입이자율은 연 5% 이다 .  리스이용자...`

**Siblings:** 형제 없음 (단독 문단)

---


## 6. 정리

In [8]:
client.close()
print("Qdrant 연결 종료")

Qdrant 연결 종료
